In [23]:
import sys 
import numpy as np
import cv2
import tensorflow as tf
from tqdm import tqdm
import sleap
from copy import copy
sys.path.append('/home/mingxiao/Desktop/jelly-sleap/sleap/jelly/python')
from animation import *
from dataset_conversion import *


In [2]:
def video_loader(video_path, target_size=None, load_as_tensor=False):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if target_size:
            gray = cv2.resize(gray, target_size)  # (width, height)
        frames.append(gray)
    
    cap.release()
    video_array = np.array(frames)  # Shape: (num_frames, height, width)
    
    if load_as_tensor:
        video_tensor = tf.constant(video_array, dtype=tf.float16) / 255.0
        video_tensor = tf.expand_dims(video_tensor, axis=-1)  # Add channel dim
        return video_tensor
    else:
        return video_array / 255.0

In [3]:
full_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_15min_reencoded_v2.mp4'
clipped_video_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_5min_track_reencoded_0.mp4'

full_video = video_loader(full_video_path)
clipped_video = video_loader(clipped_video_path)

In [4]:
pad_inx = 150 * 50 * 60
start_search_idx = 0
print(f'pad_inx: {pad_inx}')
print(f'full_video.shape: {full_video.shape}')
print(f'clipped_video.shape: {clipped_video.shape}')

pad_inx: 450000
full_video.shape: (135000, 170, 174)
clipped_video.shape: (9000, 170, 174)


In [16]:
clipped_first_frame = clipped_video[0]
clipped_last_frame = clipped_video[-1]
first_match_idx = None
last_match_idx = None
for i in tqdm(range(start_search_idx, len(full_video))):
    if np.allclose(full_video[i], clipped_first_frame, atol=0.1):
        print(i)
        match_idx = i
    elif np.allclose(full_video[i], clipped_last_frame, atol=0.12):
        print(i)
        last_match_idx = i
        # break

# print(f'match_idx: {match_idx + pad_inx}')

 23%|██▎       | 30424/135000 [00:07<00:26, 3938.76it/s]

29835
29836
29837
29838
29840
29841
29842
29843
29844
29845
29846
29850
29851
29852
29853
29854
29855
29856
29864
29865


 29%|██▉       | 39461/135000 [00:10<00:24, 3909.28it/s]

38850


100%|██████████| 135000/135000 [00:34<00:00, 3892.39it/s]


In [10]:
29835/150/60

3.315

In [17]:
38850 - 9000

29850

In [44]:
pad_inx = 150 * 50 * 60
start_idx = 29850 + pad_inx
# start_idx += 1600
end_idx = 38850 + pad_inx
print(f'start index: {start_idx}')
print(f'end index: {end_idx}')

start index: 479850
end index: 488850


In [46]:
start_idx

479850

In [10]:
a1_start_idx = 3600 * 150
print(f'a1_start_idx: {a1_start_idx}')
a1_end_idx = a1_start_idx + 3000
print(f'a1_end_idx: {a1_end_idx}')

a1_start_idx: 540000
a1_end_idx: 543000


In [76]:
# a1_indices_switch = [10, 11, 12, 13, 14, 15, 16, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
a1_indices_switch = [8, 9, 10, 11, 12, 13, 14, 15, 16, 0, 1, 2, 3, 4, 5, 6, 7]

In [4]:
indices_switch = [12, 9, 10, 8, 4, 14, 15, 16, 0, 1, 2, 5, 3, 13, 11, 7, 6]

In [5]:
a1_dataset_path = '/home/mingxiao/Desktop/jellyfish/label/animal_1_labels.v002.slp'
a1_dataset = sleap.load_file(a1_dataset_path)
print(a1_dataset)

Labels(labeled_frames=8089, videos=1, skeletons=1, tracks=0)


In [6]:
c1_dataset_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/manual_5min_c0.slp'
c1_dataset = sleap.load_file(c1_dataset_path)
print(c1_dataset)

Labels(labeled_frames=9000, videos=1, skeletons=1, tracks=17)


In [14]:
c2_dataset_path = '/home/mingxiao/Desktop/jellyfish/video/video_1_clips/correction_test/a1_1h_20s_corrected_with_scores_simplemax.slp'
c2_dataset = sleap.load_file(c2_dataset_path)
print(c2_dataset)
c2_pts = get_all_tracked_points(c2_dataset, reorder=False, interpolate=False, start_idx=0, use_labeled_only=False)
print(c2_pts.shape)

Labels(labeled_frames=3000, videos=1, skeletons=1, tracks=17)
all_tracked_points shape: (3000, 17, 2)
First non missing frame idx: 0
Missing point count: 0
(3000, 17, 2)


In [77]:
c2_pts_reorder = c2_pts[:, a1_indices_switch]
print(c2_pts_reorder.shape)
c2_pts_indices = np.arange(a1_start_idx, a1_end_idx)

(3000, 17, 2)


In [50]:
def get_coords_single_model(lf):
    for inst in lf.instances:
        if not isinstance(inst, sleap.instance.PredictedInstance):
            return inst.points_array[1:]
    raise ValueError('No labeled instance found')

def get_coords_multi_model(lf):
    coords = np.zeros((17, 2))
    if len(lf.instances) != 17:
        print(len(lf.instances))
    for inst in lf.instances:
        # if not isinstance(inst, sleap.instance.PredictedInstance):
        if True:
            idx = int(inst.track.name[6:])
            coords[idx] = inst.points_array[0]
    if not np.all(coords != 0): 
        print('some points are missing')
        return
    return coords[indices_switch]

In [78]:
import sleap.instance

labeled_indices = []
labeled_coords = []
lb_cnt = 0
start_idx = 479850
for lf in a1_dataset.labeled_frames:
    if len(lf.instances) == 2:
        lb_cnt += 1
        labeled_indices.append(lf.frame_idx)
        labeled_coords.append(get_coords_single_model(lf))
    elif len(lf.instances) == 1 and not isinstance(lf.instances[0], sleap.instance.PredictedInstance):
        lb_cnt += 1
        labeled_indices.append(lf.frame_idx)
        labeled_coords.append(get_coords_single_model(lf))
labeled_coords_arr = np.array(labeled_coords)
print(labeled_coords_arr.shape)
print(f'lb_cnt: {lb_cnt}')
printed=False
for lf in tqdm(c1_dataset.labeled_frames[1600:2500]):
    curr_frame_coords = get_coords_multi_model(lf)
    if curr_frame_coords is None:
        continue
    
    padded_idx = lf.frame_idx + start_idx
    
    if not printed:
        print(f'first frame index: {lf.frame_idx}')
        print(f'padded index: {padded_idx}')
        printed=True
    if padded_idx in labeled_indices:
        print(f'overlap at frame {lf.frame_idx}')
        corresponding_idx = labeled_indices.index(padded_idx)
        coords_1 = labeled_coords_arr[corresponding_idx]
        if not np.allclose(curr_frame_coords, coords_1, atol=5):
            labeled_coords_arr[corresponding_idx] = curr_frame_coords
            print(f'coordinates do not match at frame {lf.frame_idx}')
    else:
        labeled_indices.append(padded_idx)
        labeled_coords.append(curr_frame_coords)

for c2_idx, c2_frame_pts in zip(c2_pts_indices, c2_pts_reorder):
    if c2_idx in labeled_indices:
        print(f'overlap at frame {c2_idx}')
        corresponding_idx = labeled_indices.index(c2_idx)
        labeled_coords[corresponding_idx] = c2_frame_pts
    else:
        labeled_indices.append(c2_idx)
        labeled_coords.append(c2_frame_pts)
        
labeled_coords_arr = np.array(labeled_coords)
print(labeled_coords_arr.shape)

sorted_idx = np.argsort(labeled_indices)
labeled_indices = np.array(labeled_indices)[sorted_idx]
labeled_coords_arr = labeled_coords_arr[sorted_idx]

(1654, 17, 2)
lb_cnt: 1654


 22%|██▏       | 194/900 [00:00<00:00, 969.96it/s]

first frame index: 1600
padded index: 481450


 76%|███████▌  | 686/900 [00:00<00:00, 971.95it/s]

16
some points are missing
16
some points are missing


100%|██████████| 900/900 [00:00<00:00, 971.50it/s]


16
some points are missing
overlap at frame 541080
overlap at frame 542221
overlap at frame 542585
(5548, 17, 2)


In [56]:
479850+1600

481450

In [23]:
a1_dataset.video

Video(backend=MediaVideo(filename='C:/Users/weiss/OneDrive/Desktop/Concat_and_crop/full_video_1.avi', grayscale=True, bgr=True, dataset='', input_format=''))

In [22]:
new_5k_dataset_path = '/home/mingxiao/Desktop/jellyfish/label/multifish/multifish_animal_1_v12.slp'
new_5k_dataset = sleap.load_file(new_5k_dataset_path)

In [20]:
new_4k_dataset_path = '/home/mingxiao/Desktop/jellyfish/label/multifish/multifish_animal_1_v10.slp'
new_4k_dataset = sleap.load_file(new_4k_dataset_path)

In [21]:
new_4k_dataset.video

Video(backend=MediaVideo(filename='/home/mingxiao/Desktop/jellyfish/video/video_1_clips/sleap_full_video_1_highest_res.mp4', grayscale=True, bgr=True, dataset='', input_format=''))

In [26]:
all_tracks = [sleap.instance.Track(name=f'track_{i}', spawned_on=0) for i in range(17)]

In [79]:
new_5k_dataset.skeletons = c1_dataset.skeletons
new_5k_dataset.tracks = all_tracks
vid = new_5k_dataset.video
skl = new_5k_dataset.skeletons[0]
lbfs = []

for lbf_idx, coords_arr in zip(labeled_indices, labeled_coords_arr):
    new_instances = []
    for (x, y), trk in zip(coords_arr, all_tracks):
        point_dict = {f'tb': sleap.instance.Point(x=x, y=y)}
        tb_instance = sleap.Instance(skeleton=skl, points=point_dict, frame=lbf_idx, track=trk)
        new_instances.append(tb_instance)
    new_lf = sleap.LabeledFrame(video=vid, frame_idx=lbf_idx, instances=new_instances)
    lbfs.append(new_lf)
    
new_5k_dataset.labeled_frames = lbfs
# new_4k_dataset.tracks = []

In [61]:
new_4k_dataset_path = '/home/mingxiao/Desktop/jellyfish/label/multifish/multifish_animal_1_v11.slp'
# new_4k_dataset.save(new_4k_dataset_path)
new_4k_dataset = sleap.load_file(new_4k_dataset_path)

In [72]:
new_5k_dataset

Labels(labeled_frames=5548, videos=1, skeletons=1, tracks=17)

In [80]:
new_5k_dataset.save(new_5k_dataset_path)

In [74]:
labeled_indices[600:]

array([ 481827,  481828,  481829, ..., 3230280, 3233520, 3236760])

In [68]:
481450 in labeled_indices

True

In [70]:
2503624 in labeled_indices

False